# Step 1: Residual and Spectral Bounds for the Self-Similar Profile

Bound the profile residual, prove $Q\succeq0$, bound its maximum eigenvalue, and verify positivity of $\operatorname{sym}\nabla\overline U+Q+\eta_1I$. These are the paper's inputs `eq:45-input-EU`, `eq:45-input-Q-positive`, `eq:45-input-Q-max`, and `eq:45-input-Q-condition`.

Run from `notebooks/` with `dtype = Interval{Float64}` selected before loading `NS_Numerics.jl`.

**Inputs:** `data/external/UP.mat` and `data/external/approx_Q.mat` contain the profile and Q coefficients.

**Outputs**, under `data/generated/`:

| File | Contents / consumer |
|---|---|
| `UP_divfree.mat` | Divergence-free profile for Steps 2, 5, 6, and 7 |
| `Q_bound_radial.mat` | Normalized-radius derivative bounds for Step 4B |
| `approx_Q.mat` | Cached Q samples, with `24031 × 6031` samples per component and fifteen padding layers per side |

The generated `approx_Q.mat` contains samples, whereas the external file with the same name contains coefficients. The sample cache is created only when absent and otherwise reused. Its coefficients, scale, coordinate map, grid, and padding must match this calculation.

Read in order: coordinates and inputs → residual integral → Q derivative bounds → matrix derivative bounds → spectral bounds → compensation inside and outside Q's support. The [symmetry reference](../docs/technical_notes/symmetry_and_domains.md#profile-verification) and [quadrature notes](../docs/technical_notes/matrix_func.md) explain the domain reductions and error bounds.


In [1]:
import Pkg
Pkg.activate("..")
include("../src/NS_Numerics.jl")
using .NS_Numerics

  Activating 

Using dtype = IntervalArithmetic

project at `~/Changhe/3d-navier-stokes-nonuniqueness-main-run`


.Interval{Float64}
Number of threads = 56


## Coordinates, represented fields, and symmetries

Let $\kappa=\texttt{UP.scl\_fac}$ and $R=32$. The profile stores compactified unknowns $\widehat U,\widehat P$; the physical fields are $U=\cos\beta\,\widehat U$ and $P=\cos\beta\,\widehat P$.

| Coordinate | Definition | Domain used here |
|---|---|---|
| $r_{\rm phy}$ | Physical radius | $[0,\infty)$ |
| $\beta$ | $r_{\rm phy}=\kappa\tan\beta$ | $[0,\pi/2)$; the endpoint represents infinity |
| $s$ | Normalized radius $r_{\rm phy}/R$ | $[0,1]$ for Q's support |
| $x_Q$ | Q's stored argument $(\pi/2)s$ | $[0,\pi/2]$ |
| $\mathrm{Beta}$ | $\arctan(R/\kappa)$ | End of Q's support in β |

`interpolate(Q, ...)` receives $x_Q$. The code variable `a` is $(\pi/2)\kappa/R$, so `a*tan(β)` is $x_Q$, not physical radius. Native Q norm calls instead use `L_Q=1`; their radial derivative bounds are with respect to $s$. In contrast, `r_to_beta` receives $\kappa/R$ because its input table already differentiates in $s$.

Some arrays are reused numerically: `interpolate(Q, β_pt, θ_pt)` interprets `β_pt` as native $x_Q$ nodes, while `get_operator_matrix(UP, β_pt, θ_pt, ...)` interprets the same numbers as β. Only calls with `a*tan.(β_pt)` evaluate Q at the physical radii corresponding to that β grid.

### Why these domains suffice

E/O denote even/odd reflection at the indicated radial endpoints. Angular codes record parity at $(\theta=0,\theta=\pi/2)$, with `1` meaning E and `0` meaning O. Vector component order is `(1,2,3)`; tensor order is `(11,12,13,22,23,33)`.

| Evaluated quantity | Radial reflections | Angular codes |
|---|---|---|
| Weighted profile residual | E at β=0, O at β=π/2 | `11,00,01` |
| Physical symmetric gradient | E at β=0 and β=π/2 | `11,00,01,11,10,11` |
| Native Q | E at $x_Q=0$, O at $x_Q=\pi/2$ | `11,00,01,11,10,11` |
| Q composed with $x_Q(\beta)$ | E at zero; no reflection assumed about `Beta` | Same tensor codes |

For the first three rows, the componentwise absolute values, and those of every derivative, reflect to a full trigonometric period. This justifies the reduced domains in Bernstein estimates. Composed Q is not a finite-frequency trigonometric polynomial in β; its β bounds instead come from the chain rule.

For Q and $G=\operatorname{sym}\nabla U$, angular reflection gives $S(-\theta)=D_0S(\theta)D_0$ and $S(\pi-\theta)=D_1S(\theta)D_1$, where $D_0=\operatorname{diag}(1,-1,-1)$ and $D_1=\operatorname{diag}(1,-1,1)$. Orthogonal conjugation preserves eigenvalues, so the angular quarter domain also suffices for the matrix estimates. See the [symmetry proofs](../docs/technical_notes/symmetry_and_domains.md#domain-rules).

### Grid roles

The counts below are **interval counts**, before padding; an unpadded axis has one more sample than intervals.

| Calculation | Coordinates used by the bounds | Interior intervals | Padding per side |
|---|---|---|---|
| Initial native Q priors | $(s,\theta)\in[0,1]\times[0,\pi/2]$ | `12000 × 6000` | 0 |
| Cached Q composition and order-ten refinement | $(\beta,\theta)\in[0,\mathrm{Beta}]\times[0,\pi/2]$ | `24000 × 6000` | 15 |
| Residual quadrature | $(\beta,\theta)\in[0,\pi/2]^2$ | `18000 × 9000` | 0 |
| Order-two matrix derivative refinement | Native Q and H in $s$; gradient in β | `12000 × 6000` | 1 |
| Spectral sampling of H and Q | $(s,\theta)\in[0,1]\times[0,\pi/2]$ | `24000 × 24000` | 0 |
| Positivity sampling, each radial region | $(\beta,\theta)$ | `24000 × 24000` | 0 |

The physical radial interval is already covered by β∈[0,π/2); there is no radial doubling. Padding extends the smooth formulas for derivative stencils and is separate from any symmetry reduction.

In [2]:
# Load the stored compactified profile; the default loader imposes divergence freedom.
UP = from_mat_file("../data/external/UP.mat", 2; domain="frequency")
M0_UP, M1_UP = size(UP["u3"].freq_func)

# Persist the divergence-free-corrected profile (the div-free projection is applied
# inside from_mat_file) so later notebooks load it directly with divfree=true.
save(UP, "../data/generated/UP_divfree.mat")

# Load Q coefficients in (x_Q, θ), in tensor order 11,12,13,22,23,33.
Q = load_grad_U("../data/external/approx_Q.mat")
R = dt(32)
eta1 = dt("0.19084")
# The factor a maps tan(β) to x_Q; r_to_beta instead takes UP.scl_fac/R.
a = (Pi / dt(2)) * UP.scl_fac / R;

# Set physical β/θ lengths, normalized Q radius length L_Q=1, and the support endpoint Beta.
L0, L1, L_Q, Beta = Pi / dt(2), Pi / dt(2), one(dtype), atan(R / UP.scl_fac)

# Target derivative order for the later Q composition refinement.
k = 10;


## L² residual norm of the self-similar equation

For physical fields, write

$$
E_U=U\cdot\nabla U-L_{\rm base}(U,P),\qquad
L_{\rm base}(U,P)=\tfrac12(U+x\cdot\nabla U)+\Delta U+\nabla P.
$$

The operator functions include the radial weight $w(\beta)=\sin\beta/\cos^2\beta$, so the array `err` represents **$wE_U$**, not $E_U$ itself. Its physical norm is

$$
\|E_U\|_{L^2(\mathbb R^3)}^2
=4\pi\kappa^3\int_0^{\pi/2}\!\int_0^{\pi/2}
\sum_{i=1}^3\texttt{err}_i(\beta,\theta)^2\sin\theta\,d\theta\,d\beta.
$$

Here $2\pi$ comes from azimuthal integration, another factor 2 from equatorial symmetry, and $\kappa^3$ from radial scaling. `W_norms` bounds derivatives of the assembled weighted residual; passing it to the L² norm call enlarges the Newton–Cotes quadrature by a rigorous eighth-derivative allowance. The frequency ceilings `4*M0+10,4*M1+10` conservatively cover product terms and the component/operator frequency shifts; they are not exact maximum frequencies.

The next cell handles the residual on its unpadded grid. The separate cell after it prepares the padded Q, H, and symmetric-gradient samples for the matrix estimates.

The next block prints the residual norm as `L2_norms`; its upper endpoint gives the physical residual bound in `eq:45-input-EU`. The print label mentions the linear operator, but the expression includes convection and is the full profile residual above. See the [operator conventions](../docs/technical_notes/ns_terms.md#3-the-radial-jacobian-weight) and [weighted quadrature derivation](../docs/technical_notes/matrix_func.md#from-one-dimensional-quadrature-to-a-weighted-inner-product).

In [3]:
L2_norms, freq_lst = let
    # Unpadded physical β/θ grid for the weighted residual and its L² quadrature.
    N0, N1 = 18000, 9000
    β_pt = grid(L0, N0)
    θ_pt = grid(L1, N1)

    # Both operators return weighted physical terms, so err = w*E_U.
    lin_mat = get_operator_matrix(UP, β_pt, θ_pt, "linear")
    err = mul_A_gradB(UP, UP, β_pt, θ_pt) - apply(lin_mat, UP)

    # Conservative frequency ceilings for the weighted residual: products add bandwidths,
    # and the +10 allowance covers component and trigonometric-operator frequency shifts.
    freq_lst = [dt(4 * M0_UP + 10), dt(4 * M1_UP + 10)]
    W_norms = norm_lst(err, "W$(k)Inf"; paras=freq_lst, domain=[L0,L1])

    # Integrate the unpadded err samples with the physical measure and derivative-based quadrature allowance.
    L2_norms = norm_lst(err, "L2"; paras=W_norms)
    println("L^2 error of linear operator: ", L2_norms)
    @assert sup(L2_norms) <= inf(dt("4.969e-6"))

    (L2_norms, freq_lst)
end
GC.gc(true)


L^2 error of linear operator: [4.83829e-6, 4.9681e-6]_com


## Q derivative bounds

The first norm call bounds $Q((\pi/2)s,\theta)$ through order 12 on the native Q grid. In a returned table `W`, `W[m+1,1][i]` bounds the m-th radial derivative of component i and `W[m+1,2][i]` its m-th θ derivative. The component order is `A11,A12,A13,A22,A23,A33`.

The workflow is:

1. **Export native radial bounds.** Pack the order-0 through order-12 radial column into a `(13,6)` table in `Q_bound_radial.mat`. Step 4B reads this table and converts $s=(\kappa/R)\tan\beta$ exactly once.
2. **Convert the priors to β.** `r_to_beta` applies the chain rule on the enlarged interval specified by `β_max = 1.01*Beta`, which must cover the refinement stencils.
3. **Refine through order 10.** The order-11 and order-12 priors seed the recursive finite differences. `derivative_padding(10)=15` gives fifteen actual ghost layers per side, while `domain=[Beta,L1]` still gives the unpadded lengths and mesh spacings.
4. **Retain the bounds needed later.** Keep `Q_dββ_L∞` for the positivity estimate and the native table `Q_Wk∞` for the matrix-derivative stage. Its first five rows already supply the order-0 through order-4 priors, so they are not recomputed. The refined β table is not saved.

The second let block returns only Q_Wk∞, Q_freq_lst, and Q_dββ_L∞. Both native and padded Q sample grids remain local.

`Linf_to_paras(...; ind=1)` uses the explicit β priors, then builds the angular refinement from the refined β-order-zero and β-order-two bounds using angular Bernstein. It does not apply β Bernstein to the tangent-composed Q. See [special parameter preparation](../docs/technical_notes/matrix_func.md#special-mode-refine-β-first-then-construct-the-θ-bounds) and [recursive padding](../docs/technical_notes/matrix_func.md#why-refinement-proceeds-from-high-order-to-low-order).

See the [data guide](../docs/data.md) for the radial export.

### Frequency ceilings in the native Q coordinates

`Q_M0,Q_M1` are the largest radial and angular coefficient counts among Q's six components, independent of the profile's `M0,M1`. All current Q radial blocks use `10`, hence their largest native radial frequency is at most $2Q_{M0}-1$. The code uses the convenient upper ceiling $2Q_{M0}$; all angular families have frequencies at most $2Q_{M1}$.

Since $x_Q=(\pi/2)s$, these give the ceilings used by `Q_freq_lst`:

$$
M_s=\pi Q_{M0},\qquad M_\theta=2Q_{M1}.
$$

`norm_lst(...; paras=Q_freq_lst)` first corrects the sampled maximum to a continuous bound, then obtains derivative bounds by Bernstein powers. The native reflection symmetries above justify using the reduced rectangle. These frequencies apply to $s$ and θ, not to Q after composition with β.

In [4]:
Q_Wk∞, Q_freq_lst, Q_dββ_L∞ = let
    # Interior interval counts for the native Q grid (one more sample than intervals).
    N0, N1 = 12000, 6000

    # Here r_pt contains native x_Q nodes, not β or normalized radius s.
    r_pt = grid(L0, N0)
    θ_pt = grid(L1, N1)

    # Evaluate Q on the unpadded native grid; subsequent norm calls use s-length L_Q=1.
    Q_dict = interpolate(Q, r_pt, θ_pt);

    # Space-sample cache, distinct from the external coefficient file with the same basename.
    Q_dict_path = "../data/generated/approx_Q.mat"
    # Physical interval counts and actual padding are needed even when loading the cache.
    N0_Q, N1_Q = 24000, 6000
    padding_Q = derivative_padding(k)
    if !isfile(Q_dict_path)
    
        # Extend the physical β grid by padding_Q=15 layers per side.
        # The denominator N0_Q keeps the unpadded mesh spacing.
        β_pt_glb = grid(Beta, N0_Q, padding_Q)
    
        # Convert to Q's argument x_Q = (π/2)(UP.scl_fac/R)tan(β), with padding.
        r_pt_glb = tan.(β_pt_glb) .* a
        θ_pt_ext = grid(L1, N1_Q, padding_Q)
    
        # Cache every padded sample; the norm wrapper handles the derivative stencils.
        Q_glb_dict = interpolate(Q, r_pt_glb, θ_pt_ext)
        save(Q_glb_dict, Q_dict_path; domain="space");
    else
        # Check the cached Q sample array before using it.
        Q_glb_dict = load_grad_U(Q_dict_path; domain="space")
    end

    # Set frequency bounds and compute rigorous W^{k,∞} norms of Q for (eq. 4.45)-(eq. 4.46).
    Q_M0 = maximum(size(Q[key].freq_func, 1) for key in Q.keys)
    Q_M1 = maximum(size(Q[key].freq_func, 2) for key in Q.keys)
    Q_freq_lst = [dt(Q_M0) * Pi, dt(2 * Q_M1)]

    # Native-coordinate priors through order 12; rows 12 and 13 seed orders 11 and 12.
    # domain=[L_Q,L1] reports radial derivatives in normalized radius s.
    Q_Wk∞ = norm_lst(Q_dict, "W$(k+2)Inf"; paras=Q_freq_lst, domain=[L_Q,L1])

    # Export normalized-radius derivatives before any β conversion. Fundamental-domain
    # bounds extend to the trigonometric Q field by its reflection/periodicity symmetries.
    Q_Wk∞_radial_table = permutedims(hcat(Q_Wk∞[:, 1]...))
    save_matrix("../data/generated/Q_bound_radial.mat", Dict("Q_WkInf" => Q_Wk∞_radial_table))

    # Convert s derivatives to β derivatives on an interval covering the padded stencils.
    Q_Wk∞_beta_prior = r_to_beta(Q_Wk∞[:, 1], UP.scl_fac / R; β_max = Beta * dt(1.01))

    # Refine β first; ind=1 builds the θ correction from β-order-0/2 bounds and angular frequency.
    paras = Linf_to_paras(Q_Wk∞_beta_prior, k+2, Q_freq_lst; ind=1)

    Q_Wk∞_beta = norm_lst(Q_glb_dict, "W$(k)Inf"; paras=paras, domain=[Beta,L1], padding=padding_Q)

    # Euclidean norm of the six second-β-derivative bounds; not yet a matrix norm.
    Q_dββ_L∞ = norm(Q_Wk∞_beta[3, 1]);

    (Q_Wk∞, Q_freq_lst, Q_dββ_L∞)
end
GC.gc(true)


## Second-derivative bounds for matrix interpolation

The next cell obtains bounds needed to pass from sampled matrices to all points between samples. For the physical symmetric gradient, its radial direction is β; for native Q and $H=Q/\cos(x_Q)$, the radial direction is normalized radius $s$.

These refinements use priors through order four to obtain bounds through order two, with one padding layer per side from `derivative_padding(2)`. Ordinary `Linf_to_paras` is valid here because native Q, H, and the symmetric gradient have definite endpoint reflections in both coordinates. This differs from the earlier Q composition, which requires the special `ind=1` path.

The variables `grad_dββ_L∞`, `grad_dθθ_L∞`, `Q_drr_L∞`, `Q_dθθ_L∞`, `H_drr_L∞`, and `H_dθθ_L∞` are Euclidean norms of six componentwise derivative bounds. The matrix allowances below multiply them by $\sqrt2$ to account for off-diagonal entries appearing twice. No derivatives of eigenvalues are required.

The third let block reuses the small Q_Wk∞ table and returns the six scalar derivative bounds. Its padded Q, H, and gradient samples are local; the Q_dict samples from the preceding stage are not retained across cells.

In [5]:
grad_dββ_L∞, grad_dθθ_L∞, Q_drr_L∞, Q_dθθ_L∞, H_drr_L∞, H_dθθ_L∞ = let
    # Prepare ghost layers for the separate order-two matrix derivative refinements.
    N0, N1 = 12000, 6000
    kk = 2
    padding = derivative_padding(kk)
    β_pt = grid(L0, N0, padding)
    θ_pt = grid(L1, N1, padding)

    # For this native-Q call, the numbers in β_pt serve as x_Q nodes; no tan(β) map is applied.
    Q_dict_ext = interpolate(Q, β_pt, θ_pt)
    H_dict_ext = interpolate(Q, β_pt, θ_pt; keys=["Sec","I"])
    H_dict = copy(H_dict_ext)
    trim_border!(H_dict, padding)

    # Here the same node array is interpreted as β; this operator returns the unweighted physical gradient.
    grad_sym_mat = get_operator_matrix(UP, β_pt, θ_pt, "gradient_sym")
    grad_sym_UP_ext = apply(grad_sym_mat, UP; new_keys=["A11", "A12", "A13", "A22", "A23", "A33"]);
    grad_sym_UP = copy(grad_sym_UP_ext)
    trim_border!(grad_sym_UP, padding);

    # Refine physical symmetric-gradient bounds from order-four priors to order two in (β, θ).
    grad_norms_prior = norm_lst(grad_sym_UP, "W$(kk+2)Inf"; paras=freq_lst, domain=[L0,L1])
    paras = Linf_to_paras(grad_norms_prior, kk+2, freq_lst)
    grad_norms = norm_lst(grad_sym_UP_ext, "W$(kk)Inf"; paras=paras, domain=[L0,L1], padding=padding)
    grad_dββ_L∞, grad_dθθ_L∞ = norm(grad_norms[3, 1]), norm(grad_norms[3, 2]);

    # Refine native Q bounds in (s, θ); the same angular domain will be used after β composition.
    # Reuse orders 0–4 of the native priors already computed in the Q-bound cell.
    Q_norms_prior = Q_Wk∞[1:kk+3, :]
    paras = Linf_to_paras(Q_norms_prior, kk+2, Q_freq_lst)
    Q_norms = norm_lst(Q_dict_ext, "W$(kk)Inf"; paras=paras, domain=[L_Q,L1], padding=padding)
    # This replaces Q_dθθ_L∞ with another valid bound on the same angular derivative over Q's support.
    Q_drr_L∞, Q_dθθ_L∞ = norm(Q_norms[3, 1]), norm(Q_norms[3, 2]);

    # Refine native H = Q/cos(x_Q) bounds, with radial derivatives in s.
    H_norms_prior = norm_lst(H_dict, "W$(kk+2)Inf"; paras=Q_freq_lst, domain=[L_Q,L1])
    paras = Linf_to_paras(H_norms_prior, kk+2, Q_freq_lst)
    H_norms = norm_lst(H_dict_ext, "W$(kk)Inf"; paras=paras, domain=[L_Q,L1], padding=padding)
    H_drr_L∞, H_dθθ_L∞ = norm(H_norms[3,1]), norm(H_norms[3,2]);

    (grad_dββ_L∞, grad_dθθ_L∞, Q_drr_L∞, Q_dθθ_L∞, H_drr_L∞, H_dθθ_L∞)
end
GC.gc(true)


## Spectral bounds for H and Q

Let $S(x_0,\theta)$ be a symmetric matrix field, and let $b_0,b_\theta$ be six-vectors of componentwise supremum bounds for its second derivatives. On a grid cell, bilinear interpolation $I_hS$ satisfies

$$
\|S-I_hS\|_{\rm op}\le\|S-I_hS\|_F
\le\frac{\sqrt2}{8}\left(h_0^2\|b_0\|_2+h_\theta^2\|b_\theta\|_2\right)
=: \delta.
$$

The factor $\sqrt2$ bounds the contribution of the duplicated off-diagonal entries. The interpolant is a convex combination of corner matrices, so its largest eigenvalue is at most the largest corner eigenvalue and its smallest is at least the smallest corner eigenvalue. Weyl's inequality then gives an upper bound by **adding** δ to the sampled maximum, or a lower bound by **subtracting** δ from the sampled minimum.

The next block samples the smooth quotient $H=Q/\cos(x_Q)$, using `keys=["Sec","I"]`. At ordinary nodes `Sec` divides by $\cos(x_Q)$; at $x_Q=\pi/2$ it fills the exact modal limits $1,-3,5,\ldots$. Here $x_0=s$, and the mesh widths are `L_Q/N0` and `L1/N1`.

The minimum sampled eigenvalue of H is reduced by the allowance from `H_drr_L∞` and `H_dθθ_L∞`, giving `min_eig_H`. To bound Q, multiply the sampled eigenvalues of H by $\cos(x_Q)$, then add the allowance from `Q_drr_L∞` and `Q_dθθ_L∞`, giving `max_eig_Q`. These are matrix-entry interpolation bounds; no derivatives or Bernstein bounds of eigenvalue functions are used. Outside the support Q is zero, and its native basis vanishes at the boundary.

This fourth `let` block retains only `min_eig_H` and `max_eig_Q`. The following positivity calculation uses the same interpolation argument for minimum eigenvalues.

In [6]:
min_eig_H, max_eig_Q = let
    # Eigenvalue bounds for H=Q/cos(x) and Q.
    N0, N1 = 24000, 24000
    r_pt = grid(L0, N0)
    θ_pt = grid(L1, N1)

    eig_val = eigen_calc(interpolate(Q, r_pt, θ_pt; keys=["Sec","I"]))

    h_r, h_θ = L_Q/dt(N0), L1/dt(N1)
    min_eig_H = dt(minimum(eig_val)) -
        sqrt(dt(2))*(h_r^2*H_drr_L∞ + h_θ^2*H_dθθ_L∞)/dt(8)
    println("min_eig_H: ", min_eig_H)
    @assert inf(min_eig_H) >= 0

    max_eig_Q = dt(maximum(eig_val .* reshape(mycos.(r_pt), :, 1))) +
        sqrt(dt(2))*(h_r^2*Q_drr_L∞ + h_θ^2*Q_dθθ_L∞)/dt(8)

    println("max_eig_Q: ", max_eig_Q)
    @assert sup(max_eig_Q) <= inf(dt("4.46964"))

    (min_eig_H, max_eig_Q)
end
GC.gc(true)


min_eig_H: [8.63009e-6, 8.6301e-6]_com
max_eig_Q: [4.46962, 4.46963]_com


## Positivity compensation inside and outside Q's support

Let $G=\operatorname{sym}\nabla U$ and $Q_\beta(\beta,\theta)=Q(x_Q(\beta),\theta)$. The two grids cover

| Region | Radial interval | Matrix evaluated |
|---|---|---|
| Inner | $[0,\mathrm{Beta}]$ | $G+Q_\beta$ |
| Outer | $[\mathrm{Beta},\pi/2]$ | $G$; Q is zero outside its support |

For each region, subtract the matrix-interpolation allowance from the sampled minimum eigenvalue. In the inner region, add the allowances for G and $Q_\beta$. `Q_dββ_L∞` bounds the second β derivative of $Q_\beta$. The later assignment to `Q_dθθ_L∞` supplies a native-radius angular bound, which is equally applicable here: β∈[0,Beta] maps onto the same normalized-radius interval $s\in[0,1]$, and θ is unchanged.

The inner β grid is uniform with step $h_\beta=\mathrm{Beta}/N_0$. The Q allowance uses this β step with `Q_dββ_L∞`, and the θ step with `Q_dθθ_L∞`. The mapped points `r_pt = a*tan.(β_pt)` are used only to evaluate Q in its native argument.

For `eta1 = 0.19084`, the code retains the two regional bounds in `min_eig_parts`, then immediately prints `minimum(min_eig_parts) + eta1` and asserts that its lower endpoint is nonnegative. This verifies that adding `eta1*I` makes both compensated matrices nonnegative, as required by `eq:45-input-Q-condition`. The angular reduction follows from matrix conjugation, while the two radial intervals together cover the entire physical radial domain.

The fifth `let` block prints and returns the two bounds; its sampled fields remain local.

In [7]:
min_eig_parts = let
    values = zeros(dtype,2)
    # Uniform β grids for the inner and outer physical regions; angular conjugation permits θ∈[0,π/2].
    N0, N1 = 24000, 24000
    β_pt_lst = [grid(Beta, N0), Beta .+ grid(L0-Beta, N0)]
    θ_pt = grid(L1, N1);

    # Matrix interpolation allowances, applied in β and θ:
    # (A) the physical symmetric gradient on each region;
    # (B) the composed Q field on the inner region only.
    # The six stored components represent a symmetric 3x3 matrix. Since off-diagonal
    # entries appear twice in the full Frobenius norm, multiplying the packed six-vector
    # norm by sqrt(2) gives a conservative perturbation bound.
    eig_val = 0
    for i in 1:2
        β_pt = β_pt_lst[i]

        # Compute the symmetric part of the gradient
        grad_sym_mat = get_operator_matrix(UP, β_pt, θ_pt, "gradient_sym")
        grad_res = apply(grad_sym_mat, UP; new_keys=["A11", "A12", "A13", "A22", "A23", "A33"])
    
        # Entrywise second derivatives control the matrix interpolation error; eigenvalue derivatives are not used.
        h_β, h_θ = β_pt[2] - β_pt[1], θ_pt[2] - θ_pt[1]
        delta_eig_val = sqrt(dt(2)) * (h_β^2 * grad_dββ_L∞ + h_θ^2 * grad_dθθ_L∞) / dt(8)
    
        if i == 1
            r_pt = tan.(β_pt) .* a;  # Q's native argument x_Q, not physical radius
            # For the first part, add Q on [0, Beta]x[0, π/2]
            add_equal!(grad_res, interpolate(Q, r_pt, θ_pt))

            # Q_dββ_L∞ differentiates Q(x_Q(β),θ) in β. The maximum mapped x_Q step
            # is at least h_β for this grid, so substituting h_r only enlarges the allowance.
            delta_eig_val += sqrt(dt(2)) * (h_β^2 * Q_dββ_L∞ + h_θ^2 * Q_dθθ_L∞) / dt(8)
        end

        # Subtract the total matrix allowance; the combined check below compares with -eta1.
        eig_val = eigen_calc(grad_res)
        min_eig = minimum(eig_val) - delta_eig_val
        println("Lower bound of the minimum eigenvalue part ", i, " : ", min_eig)
        values[i] = min_eig
    end
    values
end
println("Minimum eigenvalue lower bound plus eta1: ", minimum(min_eig_parts) + eta1)
@assert inf(minimum(min_eig_parts) + eta1) >= 0
GC.gc(true)


Lower bound of the minimum eigenvalue part 1 : [-0.190835, -0.190834]_com
Lower bound of the minimum eigenvalue part 2 : [-0.172986, -0.172985]_com
Minimum eigenvalue lower bound plus eta1: [-0.000634675, -0.000634674]_com


## Verification output and paper bounds

Each interval estimate is printed before its assertion. The implemented checks imply the rounded bounds used in the paper:

| Notebook check | Paper input |
|---|---|
| `sup(L2_norms) ≤ inf(dt("4.969e-6"))` | $\varepsilon^U\le4.97\times10^{-6}$ (`eq:45-input-EU`) |
| `inf(min_eig_H) ≥ 0` | $Q\succeq0$ (`eq:45-input-Q-positive`) |
| `sup(max_eig_Q) ≤ inf(dt("4.46964"))` | $\|\lambda_{\max}(Q)\|_\infty\le4.47$ (`eq:45-input-Q-max`) |
| `inf(minimum(min_eig_parts)+eta1) ≥ 0`, `eta1=0.19084` | Compensation also holds with the paper's larger $\eta_1=0.1909$ (`eq:45-input-Q-condition`) |

The two radial regions are checked separately and then combined. `UP_divfree.mat` is saved before the final assertions; it contains the candidate coefficients rather than these scalar bounds.
